In [277]:
import os
from dotenv import load_dotenv

load_dotenv()

access_token = os.getenv('GENIUS_CLIENT_ACCESS_TOCKEN')

In [278]:
import requests

artist_name = 'The Weeknd'
song_title = 'Blinding Lights'

api_url = 'https://api.genius.com'
headers = {'Authorization': f'Bearer {access_token}'}

url = f'{api_url}/search'
params = {'q': f'{song_title} {artist_name}'}
response = requests.get(url=url, params=params, headers=headers)

hit = response.json()['response']['hits'][0]

assert hit['index'] == 'song'
assert hit['result']['primary_artist']['name'] == artist_name
assert hit['result']['title'] == song_title

song_id = hit['result']['id']
artist_id = hit['result']['primary_artist']['id']

In [279]:
import re
import string

def dom_to_plain_text(dom: dict) -> str:
    if isinstance(dom, str):
        return dom.strip()
    if dom['tag'] == 'br':
        return '\n'
    return ' '.join([dom_to_plain_text(child_dom) for child_dom in dom['children']])

def get_description(item: dict) -> str:
    description = dom_to_plain_text(item['description']['dom'])
    description = re.sub(rf'\s+([{re.escape(string.punctuation)}])', r'\1', description)
    description = re.sub(r'\s+', ' ', description)
    return description

def get_youtube_url(song: dict) -> str:
    for media in song['media']:
        if media['provider'] == 'youtube':
            return media['url']
    return None

url = f"{api_url}/songs/{song_id}"
response = requests.get(url=url, headers=headers)
song = response.json()['response']['song']

In [280]:
url = f'{api_url}/referents'
params = {'song_id': str(song_id), 'text_format': 'plain'}
response = requests.get(url=url, params=params, headers=headers)
annotations = []
for referent in response.json()['response']['referents']:
    for annotation in referent['annotations']:
        annotations += [annotation['body']['plain']]
annotations = ' '.join(annotations)

In [281]:
url = f'{api_url}/artists/{artist_id}'
response = requests.get(url=url, headers=headers)
artist = response.json()['response']['artist']

In [282]:
url = f'{api_url}/artists/{artist_id}/songs'
page = 1
songs = []
while True:
    params = {'sort': 'popularity', 'per_page': 50, 'page': page}
    response = requests.get(url=url, params=params, headers=headers)
    songs += response.json()['response']['songs']
    page = response.json()['response']['next_page']
    if not page:
        break

In [283]:
data = {
    'artist_names': song['artist_names'],
    'title': song['title'],
    'album': song['album']['name'],
    'release_date': song['release_date'],
    'language': song['language'],
    'pageviews': song['stats']['pageviews'],
    'feat_artist_count': len(song['featured_artists']),
    'translate_count': len(song['translation_songs']),
    'third_platform_count': len(song['media']),
    'has_sample': len(song['song_relationships'][0]['songs']) > 0,
    'is_cover': len(song['song_relationships'][4]['songs']) > 0,
    'is_remix': len(song['song_relationships'][6]['songs']) > 0,
    'is_translation': len(song['song_relationships'][-2]['songs']) > 0,
    'image_url': song['header_image_url'],
    'primary_color': song['song_art_primary_color'],
    'secondary_color': song['song_art_secondary_color'],
    'text_color': song['song_art_text_color'],
    'annotation_count': song['annotation_count'],
    'song_description': get_description(song),
    'artist_description': get_description(artist),
    'artist_song_count': len(songs),
    'annotations': annotations,
    'youtube_url': get_youtube_url(song),
}

In [284]:
data

{'artist_names': 'The Weeknd',
 'title': 'Blinding Lights',
 'album': 'The Highlights (Deluxe)',
 'release_date': '2019-11-29',
 'language': 'en',
 'pageviews': 3119420,
 'feat_artist_count': 0,
 'translate_count': 19,
 'third_platform_count': 3,
 'has_sample': True,
 'is_cover': False,
 'is_remix': False,
 'is_translation': False,
 'image_url': 'https://images.genius.com/34c1c35ca27a735e6e5f18611acb1c16.1000x1000x1.png',
 'primary_color': '#cc7813',
 'secondary_color': '#7c0d04',
 'text_color': '#fff',
 'annotation_count': 12,
 'song_description': "“Blinding Lights” serves as the second single for The Weeknd’s fourth studio album, After Hours. It follows the record’s lead single, “Heartless,” which was released two days prior. The track finds Abel in a constant state of distraction that he only gets relief from when in the presence of a significant other, as he sings over an up-tempo electropop instrumental that features large ‘80s-inspired synths and synthwave drums, similar to the s